In [1]:
from bs4 import BeautifulSoup
from datetime import datetime
import pandas as pd
import urllib.request
import requests
import re
import json as JSON
import sqlite3
from sqlite3 import Error
import sys
import time
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

In [2]:
def boxScoreScrape(url):
    player_data = pd.DataFrame()
    
    # res = requests.get(url)    
    # #print(res)
    # comm = re.compile("<!--|-->")
    # soup = BeautifulSoup(comm.sub("", res.text), 'html.parser')

    options = Options()
    #options.add_argument("--headless")  # Run Chrome in headless mode

    dr = webdriver.Chrome(options=options)
    dr.get(url)

    try:
        # Explicitly wait for an element to be present (adjust the selector)
        WebDriverWait(dr, 7).until(EC.presence_of_element_located((By.CLASS_NAME, "scorebox"))) # Or a more specific element.)
        ps = dr.page_source
    except Exception as e:
        print(f"Loading timed out or element not found: {e}")
        ps = dr.page_source # Get whatever page source is available.

    dr.close()
    soup = BeautifulSoup(ps, "html")
    scorebox = soup.find(attrs = {'class' : 'scorebox'})
    
    #print(scorebox)
    
    scores = scorebox.find_all(attrs = {'class' : 'score'})
    awayTeamScore = int(scores[1].getText())
    homeTeamScore = int(scores[0].getText())

    #################
    # game info #
    #################

    awayTeam = ''
    homeTeam = ''
    attendance = '0'
    stadium = 'unknown'
    city = 'unknown'
    
    gameMetadata = soup.find(attrs = {'class' : 'scorebox_meta'})
    gameMetadataList = gameMetadata.find_all('div')
    for x in gameMetadataList:
        metaitem = x.text
        #print(metaitem)
        if "Attendance" in metaitem:
            attendance = metaitem.split()[1]
        if "Venue" in metaitem:
            stadium = metaitem.split(',')[0].split()[1:]
            stadium = ' '.join(str(x) for x in stadium)
            city = metaitem.split(',')[1][1:]
        
    gDate = gameMetadataList[0].text.split()[1:4]
    startTime = gameMetadataList[0].text.split()[4]
    gDate = ' '.join(str(x) for x in gDate)
    gDate = gDate[:-1]
    date_object = datetime.strptime(gDate, "%B %d, %Y")
    fmt_date = date_object.strftime("%-m/%-d/%y")

    comp = gameMetadataList[1].text
    if comp == 'National Women\'s Soccer League (Regular Season)':
        comp = 'NWSL'

    teams = gameMetadataList[2].text.split()[:-2]
    home = teams[:teams.index('vs.')]
    away = teams[teams.index('vs.')+1:]
    
    homeTeam = ' '.join(str(x) for x in home)
    awayTeam = ' '.join(str(x) for x in away)

    
    #print(homeTeam, homeTeamScore, awayTeam, awayTeamScore)
    #print(attendance)
    #print(stadium)
    #print(city)
    #print(fmt_date)
    #print(startTime)
    #print(comp)


    gameInfoList = [fmt_date, homeTeam, homeTeamScore, awayTeamScore, awayTeam, startTime, attendance, stadium, city, comp]

    # Create the pandas DataFrame
    game_data = pd.DataFrame([gameInfoList],columns=['Date', 'Home', 'H', 'A', 'Away', 'Time', 'Attendance', 'Stadium', 'City', 'Comp'])
    game_data['Date'] = pd.to_datetime(game_data['Date'])

    pattern = re.compile(r"^stats_.*_summary$")
    player_tables = soup.find_all("table", {"id": pattern})

    homeplayertable = player_tables[0]
    awayplayertable = player_tables[1]

    player_data = pd.DataFrame()

    for i in [0,1]:

        playertable = player_tables[i]
        data_rows = playertable.find_all('tr')
        data_header = playertable.find_all('thead')
        data_header = data_header[0].find_all("tr")
        data_header = data_header[0].find_all("th")
        playerRows = [[td.getText() for td in data_rows[i].find_all(['th','td'])]
            for i in range(len(data_rows))
            ]
        playerStats = pd.DataFrame(playerRows)
    
        playerStats.columns = playerStats.iloc[1]
        playerStats.drop([0, 1, len(playerStats)-1], inplace = True)

        colnames = ['#', 'Nation', 'Age', 'Int', 'Blocks', 'xG', 'npxG', 'xAG', 'SCA', 
                        'GCA','Cmp', 'Att', 'Cmp%', 'PrgP', 'Carries', 'PrgC', 'Att', 'Succ']
        for colname in colnames:
            try:
                playerStats.drop(columns=[colname], inplace=True)
            except:
                continue
    

        if i == 0:
            playerStats.insert(2,'Team',homeTeam)
            playerStats.insert(3,'Opp',awayTeam)
        else: 
            playerStats.insert(2,'Team',awayTeam)
            playerStats.insert(3,'Opp',homeTeam)
            
        playerStats.insert(15,'Date',fmt_date)
        playerStats.insert(16,'G',1)
            
        #print(playerStats)
        player_data = pd.concat([player_data,playerStats], ignore_index=True)
        player_data['Date'] = pd.to_datetime(player_data['Date'], format="%m/%d/%y")
    
    return (game_data, player_data)

In [33]:
gameURLs= ['https://fbref.com/en/matches/4c3697c4/Olympiacos-Kerkyra-November-27-2016-Super-League-Greece',
           'https://fbref.com/en/matches/a33d8c1d/United-States-Bolivia-May-28-2018-Friendlies-M',
           'https://fbref.com/en/matches/dbd036e2/United-States-Costa-Rica-February-2-2019-Friendlies-M',
           'https://fbref.com/en/matches/4881b365/Cape-Town-City-Stellenbosch-FC-August-10-2019-Premier-Division',
           'https://fbref.com/en/matches/2105e672/United-States-Paraguay-September-16-2021-Friendlies-W',
           'https://fbref.com/en/matches/4fae253c/San-Diego-Wave-Angel-City-FC-April-2-2022-NWSL-Challenge-Cup',
           'https://fbref.com/en/matches/041bd193/San-Diego-Wave-Gotham-FC-May-7-2022-NWSL',
           'https://fbref.com/en/matches/1fe8fd04/San-Diego-Wave-Angel-City-FC-September-17-2022-NWSL',
           'https://fbref.com/en/matches/dc2606b9/San-Diego-Wave-North-Carolina-Courage-September-30-2022-NWSL',
           'https://fbref.com/en/matches/3f6bd619/United-States-Colombia-January-28-2023-Friendlies-M',
           'https://fbref.com/en/matches/119e0735/San-Diego-Wave-Chicago-Red-Stars-March-25-2023-NWSL',
           'https://fbref.com/en/matches/1a60c64d/San-Diego-Wave-North-Carolina-Courage-April-1-2023-NWSL',
           'https://fbref.com/en/matches/b06a67ca/Juventus-Napoli-April-23-2023-Serie-A',
           'https://fbref.com/en/matches/9272416d/Roma-Milan-April-29-2023-Serie-A',
           'https://fbref.com/en/matches/85f65da4/San-Diego-Wave-Portland-Thorns-FC-May-26-2023-NWSL',
           'https://fbref.com/en/matches/e229e7d5/San-Diego-Wave-OL-Reign-May-31-2023-NWSL-Challenge-Cup',
           'https://fbref.com/en/matches/70633eda/San-Diego-Wave-Angel-City-FC-June-17-2023-NWSL',
           'https://fbref.com/en/matches/e094b273/San-Diego-Wave-OL-Reign-June-24-2023-NWSL',
           'https://fbref.com/en/matches/80b3c928/United-States-Panama-July-12-2023-Gold-Cup',
           'https://fbref.com/en/matches/ac9e20a5/San-Diego-Wave-Angel-City-FC-August-5-2023-NWSL-Challenge-Cup',
           'https://fbref.com/en/matches/4366d18b/San-Diego-Wave-Gotham-FC-August-19-2023-NWSL',
           'https://fbref.com/en/matches/a1dc77b2/San-Diego-Wave-Kansas-City-Current-September-16-2023-NWSL',
           'https://fbref.com/en/matches/5cc2c4e6/San-Diego-Wave-OL-Reign-November-5-2023-NWSL',
           'https://fbref.com/en/matches/1046f207/San-Diego-Wave-Orlando-Pride-June-7-2024-NWSL',
           'https://fbref.com/en/matches/58602728/San-Diego-Wave-Chicago-Red-Stars-June-28-2024-NWSL',
           'https://fbref.com/en/matches/6122aad9/San-Diego-Wave-Angel-City-FC-August-24-2024-NWSL',
           'https://fbref.com/en/matches/abd75002/San-Diego-Wave-North-Carolina-Courage-September-8-2024-NWSL',
           'https://fbref.com/en/matches/42f9e5e4/America-FC-Juarez-February-1-2025-Liga-MX',
           'https://fbref.com/en/matches/119de659/San-Diego-FC-St-Louis-City-March-1-2025-Major-League-Soccer',
           'https://fbref.com/en/matches/0a35cfd4/San-Diego-Wave-Utah-Royals-March-22-2025-NWSL']

# missing
# 4 - Czechia vs Bulgaria 6/7/2019
# 9 - Juventus Real Madrid Rose Bowl 7/30/2022
# 22 - Juventus Milan LA 7/27/2023
# 26 - USWNT Columbia 10/29/2023
# 33 - USWNT Japan 2/26/2025

games = pd.DataFrame()
players = pd.DataFrame()
i = 1
for gameURL in gameURLs:
    print('logging game', i, '-', gameURL)
    game_data, player_data = boxScoreScrape(gameURL)
    games = pd.concat([games,game_data], ignore_index=True)
    players = pd.concat([players,player_data], ignore_index=True)
    i+=1
    time.sleep(7)

logging game 1 - https://fbref.com/en/matches/4c3697c4/Olympiacos-Kerkyra-November-27-2016-Super-League-Greece
logging game 2 - https://fbref.com/en/matches/a33d8c1d/United-States-Bolivia-May-28-2018-Friendlies-M
logging game 3 - https://fbref.com/en/matches/dbd036e2/United-States-Costa-Rica-February-2-2019-Friendlies-M
logging game 4 - https://fbref.com/en/matches/4881b365/Cape-Town-City-Stellenbosch-FC-August-10-2019-Premier-Division
logging game 5 - https://fbref.com/en/matches/2105e672/United-States-Paraguay-September-16-2021-Friendlies-W
logging game 6 - https://fbref.com/en/matches/4fae253c/San-Diego-Wave-Angel-City-FC-April-2-2022-NWSL-Challenge-Cup
logging game 7 - https://fbref.com/en/matches/041bd193/San-Diego-Wave-Gotham-FC-May-7-2022-NWSL
logging game 8 - https://fbref.com/en/matches/1fe8fd04/San-Diego-Wave-Angel-City-FC-September-17-2022-NWSL
logging game 9 - https://fbref.com/en/matches/dc2606b9/San-Diego-Wave-North-Carolina-Courage-September-30-2022-NWSL
logging game 10 

In [34]:
games

,Date,Home,H,A,Away,Time,Attendance,Stadium,City,Comp
0,2016-11-27,Olympiacos,0,0,Kerkyra,15:00,"32,115",Stadio Georgios Karaiskáki,Piraeus,Super League Greece (Matchweek 12)
1,2018-05-28,United States,3,0,Bolivia,18:30,"11,882",Talen Energy Stadium,Chester,International Friendlies (M) (Friendlies (M))
2,2019-02-02,United States,2,0,Costa Rica,12:50,"13,656",Avaya Stadium,San José,International Friendlies (M) (Friendlies (M))
3,2019-08-10,Cape Town City,1,0,Stellenbosch FC,18:00,"4,000",Cape Town Stadium,Cape Town,South African Premier Division (Matchweek 2)
4,2021-09-16,United States,9,0,Paraguay,19:30,0,FirstEnergy Stadium,Cleveland,International Friendlies (W) (Friendlies (W))
5,2022-04-02,San Diego Wave,4,2,Angel City FC,13:00,"5,158",Torero Stadium,San Diego,NWSL Challenge Cup (Group stage)
6,2022-05-07,San Diego Wave,4,0,Gotham FC,19:00,0,Torero Stadium,San Diego,NWSL
7,2022-09-17,San Diego Wave,1,0,Angel City FC,19:00,"32,000",Snapdragon Stadium,San Diego,NWSL
8,2022-09-30,San Diego Wave,0,0,North Carolina Courage,19:00,"18,000",Snapdragon Stadium,San Diego,NWSL
9,2023-01-28,United States,0,0,Colombia,16:30,"27,000",Dignity Health Sports Park,Carson,International Friendlies (M) (Friendlies (M))


In [35]:
players

1,Player,Pos,Team,Opp,Min,Gls,Ast,PK,PKatt,Sh,...,Date,G,Off,Crs,TklW,OG,PKwon,PKcon,Touches,Tkl
0,Alberto Botía,DF,Olympiacos,Kerkyra,90,0,0,0,0,0,...,2016-11-27,1,0,0,0,0,0,0,NaN,NaN
1,Andreas Bouchalakis,MF,Olympiacos,Kerkyra,56,0,0,0,0,0,...,2016-11-27,1,0,0,0,0,0,0,NaN,NaN
2,Felipe Pardo,"FW,MF",Olympiacos,Kerkyra,34,0,0,0,0,0,...,2016-11-27,1,0,0,0,0,0,0,NaN,NaN
3,Óscar Cardozo,FW,Olympiacos,Kerkyra,90,0,0,0,0,0,...,2016-11-27,1,0,0,0,0,0,0,NaN,NaN
4,Omar Elabdellaoui,"DF,MF",Olympiacos,Kerkyra,90,0,0,0,0,0,...,2016-11-27,1,0,0,0,0,0,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
914,Ana Tejada,CB,Utah Royals,San Diego Wave,90,0,0,0,0,0,...,2025-03-22,1,NaN,NaN,NaN,NaN,NaN,NaN,64,3
915,Kate Del Fava,CB,Utah Royals,San Diego Wave,90,0,0,0,0,1,...,2025-03-22,1,NaN,NaN,NaN,NaN,NaN,NaN,49,1
916,Janni Thomsen,RB,Utah Royals,San Diego Wave,89,0,0,0,0,0,...,2025-03-22,1,NaN,NaN,NaN,NaN,NaN,NaN,35,1
917,Imani Dorsey,RB,Utah Royals,San Diego Wave,1,0,0,0,0,0,...,2025-03-22,1,NaN,NaN,NaN,NaN,NaN,NaN,6,0


In [36]:
players[players['Player']=='Alex Morgan']

1,Player,Pos,Team,Opp,Min,Gls,Ast,PK,PKatt,Sh,...,Date,G,Off,Crs,TklW,OG,PKwon,PKcon,Touches,Tkl
154,Alex Morgan,AM,San Diego Wave,Angel City FC,75,2,0,0,0,5,...,2022-04-02,1,1,5,0,0,,,NaN,NaN
182,Alex Morgan,LM,San Diego Wave,Gotham FC,90,4,0,2,2,11,...,2022-05-07,1,NaN,NaN,NaN,NaN,NaN,NaN,47,2
210,Alex Morgan,FW,San Diego Wave,Angel City FC,90,0,0,0,0,1,...,2022-09-17,1,NaN,NaN,NaN,NaN,NaN,NaN,39,0
299,Alex Morgan,FW,San Diego Wave,Chicago Red Stars,90,1,0,1,1,1,...,2023-03-25,1,NaN,NaN,NaN,NaN,NaN,NaN,49,2
330,Alex Morgan,FW,San Diego Wave,North Carolina Courage,90,2,0,1,1,2,...,2023-04-01,1,NaN,NaN,NaN,NaN,NaN,NaN,32,0
422,Alex Morgan,FW,San Diego Wave,Portland Thorns FC,90,0,0,0,0,4,...,2023-05-26,1,NaN,NaN,NaN,NaN,NaN,NaN,47,0
482,Alex Morgan,FW,San Diego Wave,Angel City FC,61,0,1,0,0,0,...,2023-06-17,1,NaN,NaN,NaN,NaN,NaN,NaN,23,0
513,Alex Morgan,FW,San Diego Wave,OL Reign,90,0,1,0,0,3,...,2023-06-24,1,NaN,NaN,NaN,NaN,NaN,NaN,33,2
605,Alex Morgan,FW,San Diego Wave,Gotham FC,67,0,0,0,0,2,...,2023-08-19,1,NaN,NaN,NaN,NaN,NaN,NaN,26,2
637,Alex Morgan,FW,San Diego Wave,Kansas City Current,90,0,0,0,0,5,...,2023-09-16,1,NaN,NaN,NaN,NaN,NaN,NaN,31,1


In [39]:
skimmed = players.drop(columns=['PKatt','Fld','Off','Crs','TklW','PKwon','PKcon'])
skimmed[skimmed['Player']=='Alex Morgan']

1,Player,Pos,Team,Opp,Min,Gls,Ast,PK,Sh,SoT,CrdY,CrdR,Fls,Date,G,OG,Touches,Tkl
154,Alex Morgan,AM,San Diego Wave,Angel City FC,75,2,0,0,5,2,1,0,0,2022-04-02,1,0,NaN,NaN
182,Alex Morgan,LM,San Diego Wave,Gotham FC,90,4,0,2,11,3,0,0,NaN,2022-05-07,1,NaN,47,2
210,Alex Morgan,FW,San Diego Wave,Angel City FC,90,0,0,0,1,1,0,0,NaN,2022-09-17,1,NaN,39,0
299,Alex Morgan,FW,San Diego Wave,Chicago Red Stars,90,1,0,1,1,0,0,0,NaN,2023-03-25,1,NaN,49,2
330,Alex Morgan,FW,San Diego Wave,North Carolina Courage,90,2,0,1,2,1,0,0,NaN,2023-04-01,1,NaN,32,0
422,Alex Morgan,FW,San Diego Wave,Portland Thorns FC,90,0,0,0,4,2,0,0,NaN,2023-05-26,1,NaN,47,0
482,Alex Morgan,FW,San Diego Wave,Angel City FC,61,0,1,0,0,0,0,0,NaN,2023-06-17,1,NaN,23,0
513,Alex Morgan,FW,San Diego Wave,OL Reign,90,0,1,0,3,0,0,0,NaN,2023-06-24,1,NaN,33,2
605,Alex Morgan,FW,San Diego Wave,Gotham FC,67,0,0,0,2,1,0,0,NaN,2023-08-19,1,NaN,26,2
637,Alex Morgan,FW,San Diego Wave,Kansas City Current,90,0,0,0,5,2,0,0,NaN,2023-09-16,1,NaN,31,1
